# Làm sạch dữ liệu

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Xử lí review data

In [3]:
import dask.dataframe as dd

review_path = "/content/drive/MyDrive/amazon1/reviewdata/electronics_reviews.parquet"
reviews = dd.read_parquet(review_path)

# số review ban đầu
reviews.shape[0].compute()


43886944

In [3]:
reviews.head()
#kiểm tra kiểu dữ
print(reviews.dtypes)
# kiểm tra các cột
print(list(reviews.columns))


user_id      string[pyarrow]
asin         string[pyarrow]
rating               float64
text         string[pyarrow]
timestamp              int64
dtype: object
['user_id', 'asin', 'rating', 'text', 'timestamp']


In [4]:
# Chỉ giữ các cột cần thiết
reviews = reviews[['user_id', 'asin', 'rating', 'text', 'timestamp']]
reviews.head()


,user_id,asin,rating,text,timestamp
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B083NRGZMM,3.0,First & most offensive: they reek of gasoline ...,1658185117948
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07N69T6TM,1.0,These didn’t work. Idk if they were damaged in...,1592678549731
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B01G8JO5F2,5.0,I love these. They even come with a carry case...,1523093017534
3,AGGZ357AO26RQZVRLGU4D4N52DZQ,B001OC5JKY,5.0,I was searching for a sturdy backpack for scho...,1290278495000
4,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,B013J7WUGC,5.0,I've bought these headphones three times becau...,1676601581238


In [5]:
# Kiểm tra số lượng giá trị thiếu trên mỗi cột
print(reviews.isna().sum().compute().to_frame(name="missing_count"))


           missing_count
user_id                0
asin                   0
rating                 0
text                   0
timestamp              0


###Kiểm tra độ dài review


In [6]:
# Tạo độ dài review
reviews["text_length"] = reviews["text"].str.len().astype("int64")

# Thống kê
stats = reviews["text_length"].describe().compute()
stats = stats.apply(lambda x: f"{x:,.0f}")
stats = stats.to_frame("value")
stats


,value
count,"43,886,944"
mean,241
std,419
min,0
25%,53
50%,138
75%,319
max,"35,208"


###Lọc review không có ý nghĩa ( dưới 20 kí tự)




In [7]:
# Số review trước khi lọc
before_count = reviews.shape[0].compute()

# Lọc review rỗng và quá ngắn
reviews = reviews[reviews["text_length"] >= 20]

# Số review sau khi lọc
after_count = reviews.shape[0].compute()

print("Trước lọc:", before_count)
print("Sau lọc:", after_count)
print("Đã xóa:", before_count - after_count)


Trước lọc: 43886944
Sau lọc: 39323727
Đã xóa: 4563217


In [8]:
# Thống kê số lượng review theo từng mức rating
reviews["rating"].value_counts().compute().sort_index().to_frame("count").map(lambda x: f"{x:,}")


,count
rating,
0.0,2
1.0,"5,127,621"
2.0,"2,200,406"
3.0,"2,758,529"
4.0,"5,102,940"
5.0,"24,134,229"


In [9]:
# Số review trước khi lọc rating
before_rating = reviews.shape[0].compute()

# Lọc rating hợp lệ
reviews = reviews[(reviews["rating"] >= 1) & (reviews["rating"] <= 5)]

# Số review sau khi lọc
after_rating = reviews.shape[0].compute()
print("Trước lọc:", f"{before_rating:,}")
print("Sau lọc:", f"{after_rating:,}")
print("Đã xóa:", f"{before_rating - after_rating:,}")

Trước lọc: 39,323,727
Sau lọc: 39,323,725
Đã xóa: 2


In [10]:
#Chuẩn hoá text
reviews["text"] = reviews["text"].str.lower()



In [11]:
#Xoá ký tự đặc biệt
reviews["text"] = reviews["text"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True)


In [12]:
#Xoá khoảng trắng thừa
reviews["text"] = reviews["text"].str.strip()


In [13]:
# Convert sang datetime
reviews["timestamp"] = dd.to_datetime(
    reviews["timestamp"],
    unit="ms",
    errors="coerce"
)


In [14]:
# Đường dẫn lưu file
save_path = "/content/drive/MyDrive/amazon1/clean_reviews.parquet"

# Lưu dữ liệu đã làm sạch
reviews.to_parquet(save_path)


In [ ]:

test = dd.read_parquet(save_path)
print("Số dòng:", test.shape[0].compute())


Số dòng: 39323725


##Xử lí metadata

In [4]:
# Load tất cả file meta
path = "/content/drive/MyDrive/amazon1/metadata/raw_meta_Electronics/*.parquet"

# Dask đọc trực tiếp từ glob (ổn định hơn concat từng file)
df_meta = dd.read_parquet(path)

# Kiểm tra cấu trúc
print("Số dòng:", df_meta.shape[0].compute())
print("Số cột:", df_meta.shape[1])

df_meta.info()
df_meta.head()



Số dòng: 1610012
Số cột: 16
<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 16 entries, main_category to author
dtypes: object(5), float64(1), int64(1), string(9)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,<NA>,<NA>,<NA>
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,<NA>,<NA>,<NA>
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,<NA>,<NA>,<NA>
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,<NA>,<NA>,<NA>
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,<NA>,<NA>,<NA>





###Giữ lại các cột cần thiết

In [5]:
cols_needed = [
    'parent_asin',
    'title',
    'main_category',
    'price',
    'average_rating',
    'rating_number',
    'description'
]

df_meta = df_meta[cols_needed]
df_meta.head()



,parent_asin,title,main_category,price,average_rating,rating_number,description
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,All Electronics,None,3.5,6,[Teleporter V3 The “Teleporter V3” kit sets a ...
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,All Electronics,None,5.0,1,[HDMI In - HDMI Out]
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,Computers,19.99,4.5,246,[]
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,AMAZON FASHION,9.99,4.5,233,[]
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,Cell Phones & Accessories,14.99,3.8,64,[all Genuine High Quality Motorola Made Access...


###Xử lý giá trị thiếu

In [6]:
# Kiểm tra số lượng thiếu
df_meta.isnull().sum().compute().to_frame(name="missing_count")




,missing_count
parent_asin,0
title,0
main_category,106334
price,0
average_rating,0
rating_number,0
description,0


In [7]:
#Xử lí giá trị thiếu
df_meta['main_category'] = df_meta['main_category'].fillna('Unknown')

df_meta.isnull().sum().compute().to_frame(name="missing_count")





,missing_count
parent_asin,0
title,0
main_category,0
price,0
average_rating,0
rating_number,0
description,0


###Xử lí cột price

In [8]:
import numpy as np
import re

def parse_price(value) -> float:
    """Parse price -> float. Không hợp lệ -> NaN."""
    if value is None:
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        v = float(value)
        return v if np.isfinite(v) and v > 0 else np.nan
    text = str(value).strip().lower()
    if text in {"", "none", "nan", "null", "na", "<na>"} or "free" in text:
        return np.nan

    text = re.sub(r"[,$]", "", text)
    text = text.replace("usd", "").replace("us$", "").replace("$", "")

    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:-|to)\s*(\d+(?:\.\d+)?)", text)
    if m:
        a = float(m.group(1))
        b = float(m.group(2))
        v = (a + b) / 2.0
        return v if v > 0 else np.nan

    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return np.nan
    v = max(float(x) for x in nums)
    return v if v > 0 else np.nan

# Parse -> price_clean
df_meta["price_clean"] = df_meta["price"].apply(parse_price, meta=("price_clean", "float64"))
df_meta["price_clean"] = df_meta["price_clean"].mask(df_meta["price_clean"] <= 0, np.nan)

df_meta[["title", "price", "price_clean"]].head(5)

,title,price,price_clean
0,FS-1051 FATSHARK TELEPORTER V3 HEADSET,None,NaN
1,Ce-H22B12-S1 4Kx2K Hdmi 4Port,None,NaN
2,Digi-Tatoo Decal Skin Compatible With MacBook ...,19.99,19.99
3,NotoCity Compatible with Vivoactive 4 band 22m...,9.99,9.99
4,Motorola Droid X Essentials Combo Pack,14.99,14.99


In [9]:
# Kiểm tra sản phẩm trùng theo parent_asin
total = int(df_meta.shape[0].compute())
unique = int(df_meta["parent_asin"].nunique().compute())
print("Số dòng bị trùng:", total - unique)

# Xoá trùng, giữ lại dòng đầu tiên
df_meta = df_meta.drop_duplicates(subset=["parent_asin"])
print("Số dòng sau khi xoá trùng:", int(df_meta.shape[0].compute()))

Số dòng bị trùng: 0
Số dòng sau khi xoá trùng: 1610012


In [10]:
# Làm sạch title: bỏ khoảng trắng thừa và loại sản phẩm không có tên
df_meta["title"] = df_meta["title"].fillna("").str.strip()
df_meta = df_meta[df_meta["title"] != ""]
print("Số dòng còn lại:", int(df_meta.shape[0].compute()))

Số dòng còn lại: 1609918


In [11]:
# Convert description: list -> text, string giữ nguyên, còn lại coi như thiếu
df_meta["description"] = df_meta["description"].apply(
    lambda x: " ".join(x) if isinstance(x, list)
    else x if isinstance(x, str)
    else None,
    meta=("description", "object"),
)

missing_desc = int(df_meta["description"].isnull().sum().compute())
missing_rate = float(df_meta["description"].isnull().mean().compute())
print("Số description bị thiếu:", missing_desc)
print("Tỉ lệ description bị thiếu:", missing_rate)

Số description bị thiếu: 1609918
Tỉ lệ description bị thiếu: 1.0


In [22]:
# Lưu metadata
save_path = "/content/drive/MyDrive/amazon1/clean_metadata.parquet"

df_meta.to_parquet(save_path, write_index=False)

## Merge data